In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, ".")  # adjust if ethan_original/ isn't next to your notebook
from data_preprocessing import DataProcessor
from dnn_model import DNNModel

KAGGLE_PATH = "../../data/kaggle_dataset.csv"
ETHAN_CLEAN_PATH = "../../data/bitcoin_for_weka.csv"
BIGQUERY_PATH = "../../data/real_bitcoin_blocks_raw.csv"

In [2]:
def run_experiment(csv_path, label):
    print(f"\n{'='*60}\n{label}\n{'='*60}")

    dp = DataProcessor(csv_path)
    df = dp.load_data()
    miners_df = dp.create_miner_simulation(df)
    X_train, X_test, y_train, y_test, features = dp.prepare_training_data(miners_df)

    # Path A: his own train() + evaluate() — the partial_fit loop
    model_a = DNNModel(input_shape=X_train.shape[1])
    model_a.train(X_train, y_train, epochs=100, batch_size=16)
    test_acc = model_a.evaluate(X_test, y_test)

    # Path B: his own cross_validate() on the full data (train+test recombined)
    X_full = np.vstack([X_train, X_test])
    y_full = np.concatenate([y_train, y_test])
    model_b = DNNModel(input_shape=X_train.shape[1])
    cv_scores = model_b.cross_validate(X_full, y_full, cv=5)

    print(f"\n>>> {label}: Path A test acc = {test_acc:.4f} | Path B mean CV = {cv_scores.mean():.4f}")
    return test_acc, cv_scores.mean()

In [3]:
def prepare_bigquery_csv(src_path, dst_path):
    df = pd.read_csv(src_path)
    df = df.rename(columns={
        "block_number": "height",
        "transaction_count": "tx_count",
        "total_output_satoshis_excl_coinbase": "output_amount",
    })
    df.to_csv(dst_path, index=False)

prepare_bigquery_csv(BIGQUERY_PATH, "bigquery_renamed_temp.csv")

In [4]:
results = {}
results["Kaggle"] = run_experiment(KAGGLE_PATH, "Kaggle-format dataset")
results["Ethan cleaned"] = run_experiment(ETHAN_CLEAN_PATH, "Ethan's cleaned dataset")
results["BigQuery"] = run_experiment("bigquery_renamed_temp.csv", "BigQuery dataset")


Kaggle-format dataset
Loaded dataset with shape: (810909, 13)
Columns: ['height', 'timestamp', 'size', 'tx_count', 'difficulty', 'median_fee_rate', 'avg_fee_rate', 'total_fees', 'fee_range_min', 'fee_range_max', 'input_count', 'output_count', 'output_amount']
Features used: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
Feature statistics:
       blocks_mined  avg_transactions    avg_volume    difficulty  \
count    100.000000        100.000000    100.000000  1.000000e+02   
mean    8109.090000       1114.228251  10404.839399  7.821774e+12   
std        0.287623          6.449839    354.259323  1.976015e+09   
min     8109.000000       1099.984708   9702.485099  7.817118e+12   
25%     8109.000000       1110.411148  10135.084353  7.820612e+12   
50%     8109.000000       1114.179060  10369.657438  7.821835e+12   
75%     8109.000000       1118.075256  10683.478808  7.823337e+12   
max     8110.000

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Early stopping at epoch 27

Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Confusion Matrix:
[[10  0]
 [ 0 10]]

Stratified Cross-Validation Scores: [0.5  0.5  0.5  0.4  0.55]
Mean CV Accuracy: 0.4900 (+/- 0.0980)

>>> Kaggle-format dataset: Path A test acc = 1.0000 | Path B mean CV = 0.4900

Ethan's cleaned dataset


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.wa

Loaded dataset with shape: (810909, 6)
Columns: ['height', 'timestamp', 'size', 'tx_count', 'difficulty', 'output_amount']
Features used: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
Feature statistics:
       blocks_mined  avg_transactions    avg_volume    difficulty  \
count    100.000000        100.000000    100.000000  1.000000e+02   
mean    8109.090000       1114.228251  10404.839399  7.821774e+12   
std        0.287623          6.449839    354.259323  1.976015e+09   
min     8109.000000       1099.984708   9702.485099  7.817118e+12   
25%     8109.000000       1110.411148  10135.084353  7.820612e+12   
50%     8109.000000       1114.179060  10369.657438  7.821835e+12   
75%     8109.000000       1118.075256  10683.478808  7.823337e+12   
max     8110.000000       1129.204094  11403.203040  7.824585e+12   

            avg_fee  fee_volatility  avg_block_size         age  profitability  
cou

d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(


Epoch 30/100 - train_acc: 0.9844, val_acc: 0.9375
Early stopping at epoch 40

Test Accuracy: 0.9500

Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.90      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20


Confusion Matrix:
[[10  0]
 [ 1  9]]

Stratified Cross-Validation Scores: [0.5  0.5  0.5  0.4  0.55]
Mean CV Accuracy: 0.4900 (+/- 0.0980)

>>> Ethan's cleaned dataset: Path A test acc = 0.9500 | Path B mean CV = 0.4900

BigQuery dataset


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.wa

Loaded dataset with shape: (810909, 11)
Columns: ['number', 'timestamp', 'size', 'tx_count', 'bits', 'height', 'total_output_satoshis', 'output_amount', 'total_fee_satoshis', 'tx_count_check', 'difficulty']
Features used: ['blocks_mined', 'avg_transactions', 'avg_volume', 'difficulty', 'avg_fee', 'fee_volatility', 'avg_block_size', 'age', 'profitability']
Feature statistics:
       blocks_mined  avg_transactions    avg_volume    difficulty  \
count    100.000000        100.000000    100.000000  1.000000e+02   
mean    8109.090000       1114.224929  10404.815739  7.821704e+12   
std        0.287623          6.451771    354.243116  2.016031e+09   
min     8109.000000       1099.984708   9702.485099  7.817118e+12   
25%     8109.000000       1110.411148  10135.084353  7.820612e+12   
50%     8109.000000       1114.179060  10369.657438  7.821744e+12   
75%     8109.000000       1118.075256  10683.478808  7.823309e+12   
max     8110.000000       1129.204094  11403.203040  7.824585e+12   



d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.wa

In [5]:
print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
for label, (test_acc, cv_mean) in results.items():
    print(f"{label:20s}  Path A: {test_acc:.4f}   Path B: {cv_mean:.4f}   Gap: {test_acc - cv_mean:+.4f}")


SUMMARY
Kaggle                Path A: 1.0000   Path B: 0.4900   Gap: +0.5100
Ethan cleaned         Path A: 0.9500   Path B: 0.4900   Gap: +0.4600
BigQuery              Path A: 0.9500   Path B: 0.4800   Gap: +0.4700
